# ERP Quantum Hackathon 2026 — Original Baselines and Energy-Aware Extension — Reference Evidence Notebook

**Project:** MetalTech production scheduling | **Primary runtime:** Google Colab CPU + Qiskit Aer | **Research governance:** Human Project Manager (PM) → Agentic AI Project Coordinator (PC) → Computing Platform

**One-click workflow:** Open in Google Colab → **Runtime → Run all**. The notebook automatically installs dependencies, constructs official inputs, runs the mandatory quantum-simulator and QUBO experiments, evaluates a proposed Stage 2 extension, validates results, builds a complete evidence archive, and calls `google.colab.files.download`.

**Required real experiments:** Instance A (2 jobs × 2 machines) QAOA circuits **actually sampled using Qiskit Aer**; Instance B (3 jobs × 3 machines) **actually sampled using a binary quadratic model solver**. A classical CP-SAT reference verifies optima. Stage 2 includes two separately labeled **synthetic research extensions**: (1) downtime and deadlines, and (2) the same problem with time-of-use electricity cost. These are not organizer-supplied inputs.

**Scientific integrity:** Qiskit state-vector evaluation is used to optimize QAOA parameters; Qiskit **Aer** subsequently samples the genuine circuit. D-Wave `neal` is a **classical simulated-annealing QUBO solver**, NOT a physical QPU. All solver results are decoded and verified. Outputs are not fabricated or silently replaced with classical solutions. No account credentials, IonQ, D-Wave Leap, or VeloxQ required.

> The notebook will not upload materials to GitHub, record a video, or deliver the live talk. Those submission steps require the Human PM. Its `FINAL_RESULTS_FOR_CHATGPT.txt` is designed for straightforward pasting back into the chat.

**Unique, testable contributions:** (i) exact start-window pruning rather than an unqualified qubit-reduction claim, (ii) identical operation-level validation across genuine QAOA/Aer and a classical QUBO backend, (iii) adding time-of-use electricity cost as **linear QUBO coefficients without additional start variables**, (iv) independent full-objective optimality certificates via CP-SAT and timestamped evidence, and (v) honest reporting of when a heuristic needed a classical warm start. These are implementation features; no quantum advantage is claimed.

**What to return:** Only paste `FINAL_RESULTS_FOR_CHATGPT.txt` from the auto-downloaded ZIP after execution. Preserve the ZIP and `.ipynb` for the repository and jury. This notebook **does not** record or upload the required demo video or conduct the live presentation.

## Cell 1 — Install reproducible dependencies

This installation requires outbound internet access in the **Google Colab** runtime. No other external data files are required.

In [ ]:
%pip -q install "qiskit>=1.1,<3" "qiskit-aer>=0.14,<1" "dimod>=0.12,<1" "dwave-neal>=0.6,<1" "ortools>=9.9,<11" numpy scipy matplotlib pandas

## Cell 2 — Configuration, automatic official inputs, environment provenance

All input JSON is generated from the organizer's official table; Stage 2 downtime, deadlines and energy tariffs are separately declared as **synthetic examples**. Execution evidence is produced under `/content/ERP_QUANTUM_EVIDENCE/`.

In [ ]:
from __future__ import annotations
import os, sys, json, hashlib, platform, time, math, traceback, shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from itertools import combinations, permutations, product
import importlib.metadata as metadata
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
import dimod, neal
from ortools.sat.python import cp_model

SEED = 20261017
ROOT = Path('/content/ERP_QUANTUM_EVIDENCE') if Path('/content').exists() else Path.cwd() / 'ERP_QUANTUM_EVIDENCE'
if ROOT.exists():
    # Never silently destroy an earlier evidence run in a rerun of this notebook.
    backup_name=ROOT.name+'_prior_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    ROOT.rename(ROOT.with_name(backup_name))
    print('Previous evidence directory preserved as:', backup_name)
for folder in ['inputs','qubos','quantum','schedules','verification','figures','reports','extension','repository']:
    (ROOT/folder).mkdir(parents=True,exist_ok=True)
START_UTC = datetime.now(timezone.utc).isoformat()
RUNS = {}

def write_json(rel, obj):
    p=ROOT/rel; p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str)+'\n',encoding='utf-8')
    return p

def write_text(rel, content):
    p=ROOT/rel; p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(str(content).rstrip()+'\n',encoding='utf-8')
    return p

INSTANCES = {
    'instance_2x2': {
        'problem':'erp_scheduling',
        'jobs':[{'id':'Z1','operations':[{'machine':'T','duration':2},{'machine':'F','duration':1}]},
                {'id':'Z2','operations':[{'machine':'T','duration':1},{'machine':'F','duration':2}]}]},
    'instance_3x3': {
        'problem':'erp_scheduling',
        'jobs':[{'id':'Z1','operations':[{'machine':'T','duration':2},{'machine':'F','duration':1},{'machine':'S','duration':2}]},
                {'id':'Z2','operations':[{'machine':'T','duration':1},{'machine':'F','duration':2},{'machine':'S','duration':1}]},
                {'id':'Z3','operations':[{'machine':'T','duration':2},{'machine':'F','duration':2},{'machine':'S','duration':1}]}]}
}
for name, data in INSTANCES.items():write_json(f'inputs/{name}.json',data)
EXT_CONFIG = {
    'label':'PROPOSED_STAGE_2_NOT_OFFICIAL_INPUT',
    'source':'Demonstration manufacturing extension defined in this project',
    'mill_outage':{'machine':'F','start':3,'end':5},
    'due_dates':{'Z1':7,'Z2':5,'Z3':10},
    'priority_weights':{'Z1':1,'Z2':3,'Z3':1},
    'tardiness_multiplier':2,
    'horizon':9,
    'reference_target_F':11
}
write_json('inputs/stage2_proposed_extension.json',EXT_CONFIG)

# Synthetic, reproducible nine-hour time-of-use example inspired by the
# aerospace reference. All numerical values below are NEW demonstration
# assumptions, not official MetalTech input and not aerospace measured data.
ENERGY_EXT_CONFIG = json.loads(json.dumps(EXT_CONFIG))
ENERGY_EXT_CONFIG['label'] = 'PROPOSED_STAGE_2_ENERGY_NOT_OFFICIAL_INPUT'
ENERGY_EXT_CONFIG['energy'] = {
    'power_kw': {'T': 8, 'F': 12, 'S': 6},
    'peak_window_hours': [2, 6],
    'offpeak_cents_per_kwh': 11,
    'peak_cents_per_kwh': 27,
    'objective_weight_per_usd': 0.02,
    'assumption': 'Synthetic factory power and tariff model, 1h time grid'
}
write_json('inputs/stage2_energy_proposed_extension.json', ENERGY_EXT_CONFIG)

version_names=['qiskit','qiskit-aer','dimod','dwave-neal','ortools','numpy','scipy','matplotlib']
versions={p:metadata.version(p) for p in version_names}
ENV={'start_utc':START_UTC,'python':sys.version,'platform':platform.platform(),
     'packages':versions,'seed':SEED,'environment':'Google Colab' if 'google.colab' in sys.modules else 'Python',
     'physical_qpu_executed':False}
write_json('reports/environment.json',ENV)
print('Environment ready:',json.dumps(versions,indent=2))
print('Official input files created; Stage 2 data separately labelled.')

## Cell 3 — Scheduling QUBO, exact energy, independent validator

Binary variable `x[j,m,t]=1` means operation `(job j, machine m)` starts at integer hour `t`. Penalties enforce exactly-one start, machine non-overlap, and technological order. For Stage 2 removes milling outage-conflicting starts; both weighted tardiness and *time-of-use electricity cost* enter as linear costs on eligible starts. The energy extension adds no additional time-indexed binary variables at the same horizon.

**Fixed-horizon interpretation:** The QUBO is a feasibility test for the baseline; makespan optimality is separately certified by CP-SAT. Stage 2 fixed-horizon QUBOs optimize tardiness (and optionally energy cost) for H=9. Makespan is **not explicitly a QUBO cost term**. We independently recompute `F = makespan + 2 × weighted tardiness + 0.02 × electricity USD` (for energy) and use CP-SAT to prove the full unbounded-horizon objective. An observed QUBO sample is only called globally optimal when its recomputed F matches that certificate.

In [ ]:
class Qubo:
    def __init__(self):
        self.linear=defaultdict(float)
        self.quadratic=defaultdict(float)
        self.offset=0.0
    def var(self, name, bias=0):self.linear[name]+=float(bias)
    def pair(self, a, b, coef):
        if a==b: self.var(a,coef);return
        self.quadratic[tuple(sorted((a,b)))]+=float(coef)
    def energy(self, sample):
        return (self.offset+sum(v*int(sample.get(k,0)) for k,v in self.linear.items())
                +sum(v*int(sample.get(a,0))*int(sample.get(b,0)) for (a,b),v in self.quadratic.items()))
    def to_bqm(self):
        return dimod.BinaryQuadraticModel(dict(self.linear),dict(self.quadratic),self.offset,dimod.BINARY)

def jobs_as_dict(instance):
    return {j['id']:[(o['machine'],int(o['duration'])) for o in j['operations']]
            for j in instance['jobs']}

def electricity_cost_cents(machine, start, duration, cfg):
    """Synthetic constant-power operation, exact integer-cent accounting."""
    peak_start, peak_end = cfg['peak_window_hours']
    peak_hours = max(0, min(start + duration, peak_end) - max(start, peak_start))
    off_hours = duration - peak_hours
    kw = int(cfg['power_kw'][machine])
    return int(kw * (peak_hours * cfg['peak_cents_per_kwh'] +
                     off_hours * cfg['offpeak_cents_per_kwh']))

def peak_energy_kwh(machine, start, duration, cfg):
    peak_start, peak_end = cfg['peak_window_hours']
    return int(cfg['power_kw'][machine]) * max(
        0, min(start + duration, peak_end) - max(start, peak_start))

def build_qubo(instance,horizon,penalty=5, extension=None):
    J=jobs_as_dict(instance)
    q=Qubo(); windows={}; candidates={}
    outage=extension.get('mill_outage') if extension else None
    for job,operations in J.items():
        for k,(machine,duration) in enumerate(operations):
            earliest=sum(d for _,d in operations[:k]); latest=horizon-sum(d for _,d in operations[k:])
            assert latest>=earliest, 'Horizon below required processing chain'
            choice=[]
            for t in range(earliest,latest+1):
                if outage and machine==outage['machine'] and t<outage['end'] and t+duration>outage['start']:
                    continue
                label=f'{job}_{machine}_{t}'
                q.var(label,-penalty)
                choice.append((t,label))
                candidates[label]={'job':job,'machine':machine,'start':t,'end':t+duration}
            if not choice:
                raise ValueError(f'No possible start for {job}:{machine} at H={horizon}')
            windows[job,machine]=choice
            q.offset+=penalty
            for (_,a),(_,b) in combinations(choice,2):q.pair(a,b,2*penalty)
    operations=[(j,m,d) for j,ops in J.items() for m,d in ops]
    for (ja,ma,da),(jb,mb,db) in combinations(operations,2):
        if ma!=mb:continue
        for ta,a in windows[ja,ma]:
            for tb,b in windows[jb,mb]:
                if ta<tb+db and tb<ta+da:q.pair(a,b,penalty)
    for job,ops in J.items():
        for (ma,da),(mb,db) in zip(ops,ops[1:]):
            for ta,a in windows[job,ma]:
                for tb,b in windows[job,mb]:
                    if tb<ta+da:q.pair(a,b,penalty)
    if extension:
        for job,ops in J.items():
            last,d=ops[-1]
            for t,name in windows[job,last]:
                late=max(0,t+d-extension['due_dates'][job])
                q.var(name,extension['tardiness_multiplier']*extension['priority_weights'][job]*late)
        if extension.get('energy'):
            # The entire operation's tariff-weighted energy is a coefficient
            # on its one-hot start variable; no new binary decisions needed.
            cfg=extension['energy']
            for job, ops in J.items():
                for machine,duration in ops:
                    for t,name in windows[job,machine]:
                        cents=electricity_cost_cents(machine,t,duration,cfg)
                        q.var(name,cfg['objective_weight_per_usd'] * cents/100.0)
    return q,windows,candidates

def validate_schedule(instance,schedule,outage=None,horizon=None):
    jobs=jobs_as_dict(instance)
    expected={(j,m):d for j,ops in jobs.items() for m,d in ops}
    seen={}; issues=[]
    for op in schedule:
        key=(op['job'],op['machine'])
        if key not in expected or key in seen:issues.append(f'unknown or duplicate {key}');continue
        s,e=op['start'],op['end']
        if not(isinstance(s,int) and isinstance(e,int)) or s<0 or e-s!=expected[key]:
            issues.append(f'incorrect times for {key}')
        seen[key]=(s,e)
        if outage and op['machine']==outage['machine'] and s<outage['end'] and e>outage['start']:
            issues.append(f'outage overlap {key}')
        if horizon is not None and e>horizon:issues.append(f'horizon exceeded {key}')
    if set(seen)!=set(expected):issues.append('missing operation(s)')
    for j,ops in jobs.items():
        for (ma,_),(mb,_) in zip(ops,ops[1:]):
            if (j,ma) in seen and (j,mb) in seen and seen[j,ma][1]>seen[j,mb][0]:
                issues.append(f'precedence violated {j}: {ma}->{mb}')
    for machine in set(m for _,m in expected):
        spans=sorted((s,e,j) for (j,m),(s,e) in seen.items() if m==machine)
        for a,b in zip(spans,spans[1:]):
            if a[1]>b[0]:issues.append(f'machine conflict {machine} {a[2]} {b[2]}')
    makespan=max((e for s,e in seen.values()),default=None)
    return {'valid':not issues,'issues':issues,'makespan':makespan,'operations_checked':len(seen)}

def decode(sample,candidates):
    return sorted([dict(candidates[v]) for v in candidates if int(sample.get(v,0))==1],
                  key=lambda o:(o['start'],o['machine'],o['job']))

def full_objective(instance,schedule,ext=None):
    check=validate_schedule(instance,schedule,ext.get('mill_outage') if ext else None)
    if not check['valid']:return None
    if not ext:return {'makespan':check['makespan']}
    weighted=0; lateness={}
    jobs=jobs_as_dict(instance)
    for j,ops in jobs.items():
        last=ops[-1][0]
        completion=next(o['end'] for o in schedule if o['job']==j and o['machine']==last)
        late=max(0,completion-ext['due_dates'][j]);lateness[j]=late
        weighted+=late*ext['priority_weights'][j]
    out={'makespan':check['makespan'],'tardiness_by_job':lateness,
         'weighted_tardiness':weighted,
         'F':check['makespan']+ext['tardiness_multiplier']*weighted}
    if ext.get('energy'):
        cfg=ext['energy']
        cents=sum(electricity_cost_cents(o['machine'],o['start'],
                  o['end']-o['start'],cfg) for o in schedule)
        peak_kwh=sum(peak_energy_kwh(o['machine'],o['start'],
                     o['end']-o['start'],cfg) for o in schedule)
        out.update({'electricity_cost_cents':int(cents),
            'electricity_cost_usd':round(cents/100.0,2),
            'peak_energy_kwh':peak_kwh,
            'energy_objective_contribution':cfg['objective_weight_per_usd']*cents/100.0})
        out['F']+=cfg['objective_weight_per_usd']*cents/100.0
    return out

def save_qubo(name,q,windows):
    labels=list(q.linear)
    write_json(f'qubos/{name}_model.json',{'vartype':'BINARY','offset':q.offset,
       'linear':dict(q.linear),'quadratic':[{'u':u,'v':v,'bias':b} for (u,v),b in q.quadratic.items()],
       'start_windows':{f'{j}_{m}':[t for t,n in choices] for (j,m),choices in windows.items()},
       'variable_count':len(labels),'interaction_count':len(q.quadratic)})
    return {'variables':len(labels),'interactions':len(q.quadratic),'offset':q.offset}

qa,wa,ca=build_qubo(INSTANCES['instance_2x2'],4,penalty=1)
qb,wb,cb=build_qubo(INSTANCES['instance_3x3'],8,penalty=5)
qe,we,ce=build_qubo(INSTANCES['instance_3x3'],9,penalty=100,extension=EXT_CONFIG)
qenergy,wenergy,cenergy=build_qubo(INSTANCES['instance_3x3'],9,penalty=100,extension=ENERGY_EXT_CONFIG)
MA=save_qubo('instance_2x2_H4',qa,wa)
MB=save_qubo('instance_3x3_H8',qb,wb)
ME=save_qubo('stage2_H9',qe,we)
MENERGY=save_qubo('stage2_energy_H9',qenergy,wenergy)
assert MA['variables']==8 and MB['variables']==39 and ME['variables']==40 and MENERGY['variables']==40
assert ME['interactions']==MENERGY['interactions'], 'Linear energy cost must not add pairwise terms'
assert MA['interactions']==12 and MB['interactions']==181
print('QUBO construction PASS:',{'A':MA,'B':MB,'Stage2':ME,'Stage2Energy':MENERGY})

## Cell 4 — Certify the exact classical optima with OR-Tools CP-SAT

This is the **independent mathematical reference**, not a quantum result. A CP-SAT `OPTIMAL` status is required before we claim global optimality. For the synthetic energy extension, CP-SAT represents per-operation tariff cost by an exact integer-cent start-time lookup (AddElement). Objective scaling by 5,000 preserves the specified weight of 0.02 per dollar: `5000*Cmax + 10000*weighted_tardiness + electricity_cost_cents`.

In [ ]:
def exact_cp_sat(instance,extension=None,max_seconds=60,horizon_limit=None):
    J=jobs_as_dict(instance); H=sum(d for ops in J.values() for m,d in ops)+10
    model=cp_model.CpModel(); intervals=defaultdict(list); starts={}; ends={}
    for job,ops in J.items():
        for machine,duration in ops:
            s=model.NewIntVar(0,H-duration,f's_{job}_{machine}')
            e=model.NewIntVar(0,H,f'e_{job}_{machine}')
            interval=model.NewIntervalVar(s,duration,e,f'i_{job}_{machine}')
            starts[job,machine]=s; ends[job,machine]=e;intervals[machine].append(interval)
        for (ma,_),(mb,_) in zip(ops,ops[1:]):model.Add(starts[job,mb]>=ends[job,ma])
    if extension:
        o=extension['mill_outage']; s=o['start'];duration=o['end']-s
        outage_var=model.NewIntervalVar(s,duration,o['end'],'outage_interval')
        intervals[o['machine']].append(outage_var)
    for machine,ivs in intervals.items():model.AddNoOverlap(ivs)
    cmax=model.NewIntVar(0,H,'makespan')
    model.AddMaxEquality(cmax,[ends[j,ops[-1][0]] for j,ops in J.items()])
    if horizon_limit is not None: model.Add(cmax <= horizon_limit)
    if not extension:model.Minimize(cmax)
    else:
        t=[]
        for j,ops in J.items():
            lateness=model.NewIntVar(0,H,'late_'+j)
            model.AddMaxEquality(lateness,[ends[j,ops[-1][0]]-extension['due_dates'][j],0])
            t.append(extension['priority_weights'][j]*lateness)
        if extension.get('energy'):
            # 5000*(Cmax + 2*weighted_tardiness + 0.02*cost_USD)
            # = 5000*Cmax + 10000*weighted_tardiness + cost_cents.
            # Direct AddElement on each start time is an exact linear-table
            # model for integer-hour operation starts and synthetic tariffs.
            cfg=extension['energy']; costs=[]
            assert cfg['objective_weight_per_usd']==0.02
            for job,ops in J.items():
                for machine,duration in ops:
                    table=[electricity_cost_cents(machine,s,duration,cfg)
                           for s in range(H-duration+1)]
                    c=model.NewIntVar(min(table),max(table),f'cost_{job}_{machine}')
                    model.AddElement(starts[job,machine],table,c)
                    costs.append(c)
            model.Minimize(5000*cmax + 5000*extension['tardiness_multiplier']*sum(t)
                           + sum(costs))
        else:
            model.Minimize(cmax+extension['tardiness_multiplier']*sum(t))
    solver=cp_model.CpSolver()
    solver.parameters.max_time_in_seconds=max_seconds
    solver.parameters.random_seed=SEED%2147483647
    solver.parameters.num_search_workers=1
    t0=time.perf_counter();status=solver.Solve(model);elapsed=time.perf_counter()-t0
    res={'status':solver.StatusName(status),'optimal':status==cp_model.OPTIMAL,
         'runtime_seconds':elapsed,'objective':(solver.ObjectiveValue()/5000.0 if extension and extension.get('energy') else solver.ObjectiveValue()) if status in (cp_model.FEASIBLE,cp_model.OPTIMAL) else None}
    if status in (cp_model.FEASIBLE,cp_model.OPTIMAL):
        schedule=[{'job':j,'machine':m,'start':int(solver.Value(starts[j,m])),
                   'end':int(solver.Value(ends[j,m]))} for j,ops in J.items() for m,d in ops]
        res['schedule']=sorted(schedule,key=lambda o:(o['start'],o['machine'],o['job']))
        res['validation']=validate_schedule(instance,res['schedule'],extension.get('mill_outage') if extension else None)
        res['objective_recomputed']=full_objective(instance,res['schedule'],extension)
        if extension: assert abs(res['objective_recomputed']['F']-res['objective'])<1e-7, 'CP-SAT objective-accounting mismatch'
    return res

references={}
for key,inst,ext,target in [('A',INSTANCES['instance_2x2'],None,4),
                            ('B',INSTANCES['instance_3x3'],None,8),
                            ('Stage2',INSTANCES['instance_3x3'],EXT_CONFIG,11),
                            ('Stage2Energy',INSTANCES['instance_3x3'],ENERGY_EXT_CONFIG,None)]:
    r=exact_cp_sat(inst,ext);references[key]=r
    write_json(f'verification/CP_SAT_{key}.json',r)
    assert r['optimal'] and r['validation']['valid'], (key,r)
    if target is not None: assert int(round(r['objective']))==target, (key,r)
    if key=='Stage2Energy': assert r['objective_recomputed']['F']>=11, 'Energy objective must include original Stage2 cost'
    print('CP-SAT certified',key,'objective',r['objective'],'makespan',r['validation']['makespan'])

# Separate horizon-constrained reference for the optional labelled warm start.
# Prevents incorrectly seeding a 9-hour QUBO with a longer valid global solution.
r_h9=exact_cp_sat(INSTANCES['instance_3x3'],EXT_CONFIG,horizon_limit=EXT_CONFIG['horizon'])
assert r_h9['optimal'] and r_h9['objective']==11 and r_h9['validation']['makespan']<=9
references['Stage2H9']=r_h9
write_json('verification/CP_SAT_Stage2_H9.json',r_h9)
print('Stage2 fixed-H9 reference verified for optional warm-start')

## Cell 5 — Instance A: actual QAOA circuits and measurements on Qiskit Aer

QAOA parameters are fitted by deterministic multi-start variational optimization using **Qiskit `Statevector`** for exact expected energy. Then the *actual QAOA circuit* is transpiled and executed using **Qiskit Aer** with 4,096 shots, and decoded measurement bitstrings are verified. This distinction is preserved in the report.

In [ ]:
A_LABELS=list(qa.linear.keys());N_A=len(A_LABELS)
linear_vec=np.array([qa.linear[k] for k in A_LABELS],dtype=float)
index_a={name:i for i,name in enumerate(A_LABELS)}
Q_Z={i:-float(qa.linear[name])/2 for i,name in enumerate(A_LABELS)}
Q_ZZ={}
for (name_i,name_j),coef in qa.quadratic.items():
    i=index_a[name_i];j=index_a[name_j];z=coef/4
    Q_Z[i]-=z;Q_Z[j]-=z;Q_ZZ[i,j]=z
ENERGIES_A=np.array([qa.energy({name:(bits>>i)&1 for i,name in enumerate(A_LABELS)})
                     for bits in range(1<<N_A)],dtype=float)
solutions_a=np.where(np.isclose(ENERGIES_A,ENERGIES_A.min(),atol=1e-9))[0].tolist()
assert min(ENERGIES_A)==0 and len(solutions_a)==1
write_json('quantum/A_exact_diagonal_ground_state.json',{
    'variables':A_LABELS,'energy_min':float(ENERGIES_A.min()),
    'optimal_integer_bitstrings':solutions_a,
    'optimal_big_endian_strings':[format(k,f'0{N_A}b') for k in solutions_a]
})

def create_qaoa(params,depth,measure=False):
    qc=QuantumCircuit(N_A)
    qc.h(range(N_A))
    for layer in range(depth):
        gamma,beta=params[layer],params[depth+layer]
        for i,h in Q_Z.items():
            if abs(h)>1e-12:qc.rz(2*gamma*h,i)
        for (i,j),h in Q_ZZ.items():
            if abs(h)>1e-12:qc.rzz(2*gamma*h,i,j)
        for i in range(N_A):qc.rx(2*beta,i)
    if measure:qc.measure_all()
    return qc

def expected_energy(x,p):
    psi=Statevector.from_instruction(create_qaoa(x,p))
    probabilities=np.abs(psi.data)**2
    return float(np.dot(probabilities,ENERGIES_A))

qaoa_trials=[];best_trial=None
start_fit=time.perf_counter()
for depth in (1,2,3):
    for run in range(3):
        rng=np.random.default_rng(SEED+100*depth+run)
        x0=np.r_[rng.uniform(0,2*np.pi,depth),rng.uniform(0,np.pi,depth)]
        result=minimize(lambda x:expected_energy(x,depth),x0,method='COBYLA',
                        options={'maxiter':105,'rhobeg':0.5,'tol':0.0003})
        item={'depth':depth,'restart':run,'expected_energy':float(result.fun),
              'angles':result.x.tolist(),'nfev':int(result.nfev),'optimizer_success':bool(result.success)}
        qaoa_trials.append(item)
        if best_trial is None or item['expected_energy']<best_trial['expected_energy']:best_trial=item
        print('QAOA fit p',depth,'seed',run,'energy',round(result.fun,6))
fit_seconds=time.perf_counter()-start_fit
write_json('quantum/A_QAOA_optimizer_trials.json',qaoa_trials)

qc_meas=create_qaoa(np.array(best_trial['angles']),best_trial['depth'],measure=True)
simulator=AerSimulator(method='statevector',seed_simulator=SEED)
transpiled=transpile(qc_meas,simulator,optimization_level=1,seed_transpiler=SEED)
shot_count=4096
t0=time.perf_counter()
raw_counts=simulator.run(transpiled,shots=shot_count,seed_simulator=SEED).result().get_counts()
aer_runtime=time.perf_counter()-t0
counts={str(k).replace(' ',''):int(v) for k,v in raw_counts.items()}
write_json('quantum/A_QAOA_Aer_counts.json',{'counts':counts,'shots':shot_count,
 'backend':'Qiskit AerSimulator(method=statevector)','depth':best_trial['depth'],'best_fit':best_trial,
 'fit_seconds':fit_seconds,'aer_sampling_seconds':aer_runtime})
try:
    from qiskit import qasm3
    write_text('quantum/A_QAOA_final_circuit.qasm3',qasm3.dumps(qc_meas))
except Exception as exc:
    # Circuit still exists as an executable object even if the installed exporter changes.
    write_text('quantum/A_QAOA_final_circuit.txt',str(qc_meas))
    write_text('quantum/A_QASM3_export_warning.txt',str(exc))

a_candidates=[]
for bitstr,num in counts.items():
    bits=int(bitstr,2)
    sample={v:(bits>>i)&1 for i,v in enumerate(A_LABELS)}
    schedule=decode(sample,ca)
    validation=validate_schedule(INSTANCES['instance_2x2'],schedule, horizon=4)
    a_candidates.append({'bitstring':bitstr,'frequency':num,'energy':qa.energy(sample),
                         'validation':validation,'schedule':schedule})
a_candidates.sort(key=lambda r:(not r['validation']['valid'],r['energy'],-r['frequency']))
a_valid=[r for r in a_candidates if r['validation']['valid'] and r['validation']['makespan']==4]
a_optimal_shots=sum(r['frequency'] for r in a_valid)
ra={'status':'PASS' if a_optimal_shots>0 else 'NO_OPTIMAL_SAMPLE',
    'backend':'Qiskit Aer simulator','algorithm':'QAOA',
    'optimization':'Qiskit Statevector + SciPy COBYLA multi-start',
    'depth':best_trial['depth'],'shots':shot_count,
    'optimal_shots':a_optimal_shots,'optimal_shot_fraction':a_optimal_shots/shot_count,
    'fit_seconds':fit_seconds,'aer_sampling_seconds':aer_runtime,
    'selected_candidate':a_valid[0] if a_valid else None,
    'observed_unique_bitstrings':len(counts)}
write_json('quantum/A_QAOA_summary.json',ra)
RUNS['A']=ra
if a_valid:
    write_json('schedules/schedule_2x2.json',{'instance':'instance_2x2',
        'method':'QAOA (Qiskit Aer state-vector simulator)', 'makespan':4,'schedule':a_valid[0]['schedule']})
else:print('WARNING: No valid four-hour quantum sample; do not claim Instance A completed.')

fig,ax=plt.subplots(figsize=(10,4)); top=sorted(counts.items(),key=lambda x:x[1],reverse=True)[:18]
ax.bar([k for k,n in top],[n for k,n in top]);ax.tick_params(axis='x',rotation=70)
ax.set_title('QAOA on Qiskit Aer — observed bitstrings');ax.set_ylabel('Measurement counts')
fig.tight_layout();fig.savefig(ROOT/'figures/A_QAOA_counts.png',dpi=160);plt.close(fig)
print('INSTANCE A',ra['status'],'optimal shots',a_optimal_shots,'/',shot_count)

## Cell 6 — Instance B: QUBO solver execution and independent verification

`neal` is an **actual, classical binary quadratic solver**. It is not physical D-Wave hardware. We first try unseeded annealing. If a valid optimum is not found, we explicitly retry with a CP-SAT-derived warm-start, which is labelled separately (not misrepresented as unseeded discovery).

In [ ]:
SAMPLER=neal.SimulatedAnnealingSampler()
def execute_neal(label, instance, q, cand, horizon, ext=None, ref_key=None, reads=1000, sweeps=1500):
    bqm=q.to_bqm();time_start=time.perf_counter()
    batch=SAMPLER.sample(bqm,num_reads=reads,num_sweeps=sweeps,seed=SEED%2147483647)
    records=[]
    def ingest(samples,method):
        for row in samples.data(['sample','energy','num_occurrences']):
            sample=dict(row.sample);schedule=decode(sample,cand)
            validation=validate_schedule(instance,schedule,ext.get('mill_outage') if ext else None,horizon)
            objective=full_objective(instance,schedule,ext) if validation['valid'] else None
            en=q.energy(sample)
            if abs(en-float(row.energy))>1e-5:raise RuntimeError('BQM QUBO energy mismatch')
            records.append({'source':method,'energy':en,'occurrences':int(row.num_occurrences),
                            'validation':validation,'objective':objective,'schedule':schedule})
    ingest(batch,'unseeded_neal')
    feasible=[r for r in records if r['validation']['valid']]
    warmstart_used=False
    if not feasible:
        # Fallback actually executes the BQM sampler. It is honestly identified as classical warm-start.
        ref=references[ref_key]
        seed_sample={name:0 for name in q.linear}
        for op in ref['schedule']:
            key=f"{op['job']}_{op['machine']}_{op['start']}"
            if key not in seed_sample:raise RuntimeError('Reference schedule cannot be encoded at fixed horizon')
            seed_sample[key]=1
        warmstart_used=True
        batch2=SAMPLER.sample(bqm,num_reads=200,num_sweeps=1200,seed=(SEED+1)%2147483647,
                 initial_states=[seed_sample],initial_states_generator='tile')
        ingest(batch2,'CP_SAT_warm_started_neal')
        feasible=[r for r in records if r['validation']['valid']]
    candidates_sorted=sorted(feasible,key=lambda r:(r['objective']['F'] if ext else r['validation']['makespan'],r['energy']))
    best=candidates_sorted[0] if candidates_sorted else None
    output={'status':'PASS' if best else 'NO_FEASIBLE_QUBO_SAMPLE',
            'solver':'neal.SimulatedAnnealingSampler (classical QUBO)',
            'initial_reads':reads,'sweeps':sweeps,'warmstart_used':warmstart_used,
            'wall_seconds':time.perf_counter()-time_start,
            'samples_examined':len(records),'feasible_sample_rows':len(feasible),
            'best':best,
            'top_samples':[{'source':r['source'],'energy':r['energy'],'occurrences':r['occurrences'],
                'valid':r['validation']['valid'],'objective':r['objective']}
                for r in sorted(records,key=lambda r:r['energy'])[:15]]}
    write_json(f'reports/{label}_solver_report.json',output)
    return output

rb=execute_neal('B',INSTANCES['instance_3x3'],qb,cb,8,None,'B',1000,1800)
RUNS['B']=rb
assert references['B']['optimal']
if rb['best'] and rb['best']['validation']['makespan']==8:
    write_json('schedules/schedule_3x3.json',{'instance':'instance_3x3',
      'method':'Classical QUBO solver (neal simulated annealing'+
       ('; CP-SAT warm start' if rb['best']['source']!='unseeded_neal' else '')+')',
      'makespan':8,'schedule':rb['best']['schedule']})
print('INSTANCE B',rb['status'],'energy',rb['best']['energy'] if rb['best'] else None,
      'warm-start?',rb['warmstart_used'])

## Cell 7 — Stage 2: actual QUBO solver, downtime, deadlines, CP-SAT benchmark

**Proposed experiment**, not an official challenge instance: milling unavailable in `[3,5)`; deadlines Z1=7, Z2=5, Z3=10; weights 1,3,1; objective `F = Cmax + 2·weighted tardiness`. A horizon-9 QUBO optimizes weighted tardiness. The globally optimal value of F is separately established by CP-SAT over a sufficient general scheduling horizon.

In [ ]:
re=execute_neal('Stage2',INSTANCES['instance_3x3'],qe,ce,9,EXT_CONFIG,'Stage2H9',1400,2000)
RUNS['Stage2']=re
if re['best']:
    ext_obj=re['best']['objective']
    write_json('extension/stage2_schedule.json',{'instance':'proposed_stage2','method':re['best']['source']+
      ' (neal classical QUBO)','objective':ext_obj,'schedule':re['best']['schedule'],
      'valid':re['best']['validation']['valid'],'proposed_parameters':EXT_CONFIG})
print('STAGE 2',re['status'],'best objective',re['best']['objective'] if re['best'] else None,
      'CP-SAT certified F',references['Stage2']['objective'])

## Cell 8 — **Stage 2 upgrade: energy-aware manufacturing QUBO**

**Problem:** The original Stage 2 schedule must respect milling downtime and customer due dates, but different *feasible* operation starts can draw electricity at different tariff rates. Optimize scheduling with time-of-use electricity cost and retain full feasibility.

**Our distinctive approach:** For each operation and feasible start time, calculate its exact electricity charge from a synthetic constant machine power (T 8 kW, F 12 kW, S 6 kW). Peak electricity costs 27¢/kWh during hours `[2,6)` and off-peak 11¢/kWh. The weighted charge **enters only as a linear coefficient of the existing one-hot start variable**, so the energy-aware QUBO has the *same 40 variables and same quadratic interaction count* as the baseline Stage 2 QUBO at H=9. We do not claim novelty for time-indexed energy-cost encoding itself; our contribution is a transparent, auditable implementation with exact equivalence checks and evidence.

**Mathematical objective:** `F = Cmax [hours] + 2 Σ_j w_j T_j [weighted hours] + 0.02 × electricity_cost_USD`. We minimize tardiness plus weighted energy inside a fixed H=9 QUBO, then independently evaluate the **full** F and compare against an exact CP-SAT optimum. These are *investigator-chosen synthetic figures*, not parameters from the hackathon PDF or turbine-blade reference. The nine-hour tariff window is illustrative, not a literal five-day aerospace tariff schedule.

**Expected evidence:** The model has 40 variables; 39-variable official Instance B stays unchanged. This experiment qualifies as globally optimal only if its audited F matches the exact CP-SAT reference to numerical tolerance. Otherwise the notebook reports the gap.

In [ ]:
# Real classical QUBO solver execution; physical QPU use remains disabled.
energy_run=execute_neal('Stage2Energy',INSTANCES['instance_3x3'],
    qenergy,cenergy,9,ENERGY_EXT_CONFIG,'Stage2Energy',reads=1400,sweeps=2000)
RUNS['Stage2Energy']=energy_run
energy_reference=references['Stage2Energy']['objective']
energy_best=energy_run['best']
energy_pass=(energy_best is not None and
             abs(energy_best['objective']['F']-energy_reference)<1e-7)
if energy_best:
    write_json('extension/stage2_energy_schedule.json',{
        'instance':'proposed_stage2_energy',
        'method':energy_best['source']+' (neal classical binary quadratic sampler)',
        'objective':energy_best['objective'],
        'schedule':energy_best['schedule'],
        'validation':energy_best['validation'],
        'is_cp_sat_global_optimum':energy_pass,
        'synthetic_input':ENERGY_EXT_CONFIG})
print('ENERGY-AWARE STAGE2:', 'CP-SAT GLOBAL OPTIMUM MATCHED' if energy_pass
      else 'FEASIBLE BUT NOT GLOBALLY CERTIFIED' if energy_best else 'NO QUBO FEASIBLE SAMPLE')
# Audited comparison on identical synthetic tariff parameters; may show no savings.
prior_schedule_energy=(full_objective(INSTANCES['instance_3x3'],
    re['best']['schedule'],ENERGY_EXT_CONFIG) if re['best'] else None)
energy_comparison={
    'baseline_downtime_deadline_sampler_on_energy_tariff':prior_schedule_energy,
    'energy_aware_sampler':energy_best['objective'] if energy_best else None,
    'same_synthetic_tariff':ENERGY_EXT_CONFIG['energy'],
    'energy_cost_delta_usd_baseline_minus_energy':(
        round(prior_schedule_energy['electricity_cost_usd']-
              energy_best['objective']['electricity_cost_usd'],2)
        if prior_schedule_energy and energy_best else None),
    'interpretation':'Difference is an empirical result, not a promised saving or quantum advantage.'
}
write_json('extension/stage2_energy_comparison.json',energy_comparison)
print('CP-SAT objective',round(energy_reference,6),
      'QUBO candidate objective',
      round(energy_best['objective']['F'],6) if energy_best else 'None',
      'QUBO variables',MENERGY['variables'])


## Cell 9 — Plot schedules and construct verifiable, compact final summary

Only results from actual sampler returns qualify as solver outcomes. If a method fails, it is prominently marked; no classical reference is substituted as though generated by that method.

In [ ]:
PALETTE={'Z1':'#2076A3','Z2':'#27B5B7','Z3':'#EE9B52'}
def gantt_chart(schedule,filename,title,outage=None):
    machines=sorted({o['machine'] for o in schedule},key=lambda m:['T','F','S'].index(m))
    fig,ax=plt.subplots(figsize=(9,3))
    for k,m in enumerate(machines):
        if outage and m==outage['machine']:
            ax.broken_barh([(outage['start'],outage['end']-outage['start'])],(k-.33,.66),
                           facecolors='#e5a339',alpha=.25,hatch='//',edgecolors='#bd7d00')
        for op in schedule:
            if op['machine']!=m:continue
            ax.broken_barh([(op['start'],op['end']-op['start'])],(k-.3,.6),facecolors=PALETTE[op['job']])
            ax.text((op['start']+op['end'])/2,k,op['job'],ha='center',va='center',color='white',weight='bold',fontsize=9)
    ax.set_yticks(range(len(machines)),machines)
    ax.set_xlabel('Hours from t=0');ax.set_title(title);ax.set_ylim(-.7,len(machines)-.3)
    ax.invert_yaxis();ax.grid(axis='x',alpha=.18);fig.tight_layout()
    fig.savefig(ROOT/'figures'/filename,dpi=160);plt.close(fig)

for key,run,filename in [('A',ra,'A_QAOA_Gantt.png'),('B',rb,'B_QUBO_Gantt.png'),('Stage2',re,'Stage2_Gantt.png'),('Stage2Energy',energy_run,'Stage2_Energy_Gantt.png')]:
    candidate=run.get('selected_candidate') if key=='A' else run.get('best')
    if candidate:
        gantt_chart(candidate['schedule'],filename,f'{key}: independently verified solver sample',
                    EXT_CONFIG['mill_outage'] if key in ('Stage2','Stage2Energy') else None)

qualifications={
  'A_qaoa_optimal':ra['status']=='PASS',
  'B_qubo_optimal':rb['status']=='PASS' and bool(rb['best']) and rb['best']['validation']['makespan']==8,
  'Stage2_global_optimal':bool(re['best']) and abs(re['best']['objective']['F']-references['Stage2']['objective'])<1e-7,
  'Stage2Energy_global_optimal':energy_pass,
  'classical_certificates':all(r['optimal'] for r in references.values())
}
write_json('verification/acceptance_gates.json',qualifications)
summary=[
 'QUANTUM HACKATHON 2026 — FINAL RESULTS FOR CHATGPT',
 'Generated UTC: '+datetime.now(timezone.utc).isoformat(),
 'Mode: Google Colab CPU, Qiskit Aer simulator + neal classical QUBO + OR-Tools CP-SAT',
 'NO physical quantum processor was used. Stage 2 inputs are proposed (not official).',
 '=== INSTANCE A (OFFICIAL 2x2) ===',
 f'CP-SAT exact makespan: {references["A"]["objective"]} h (OPTIMAL)',
 f'QUBO variables/interactions: {MA["variables"]}/{MA["interactions"]}',
 f'QAOA on Aer status: {ra["status"]}',
 f'QAOA selected depth: {ra["depth"]}; shots: {ra["shots"]}; optimal shots: {ra["optimal_shots"]}; optimal shot fraction: {ra["optimal_shot_fraction"]:.6f}',
 f'QAOA fit and Aer sampling times, seconds: {ra["fit_seconds"]:.3f}, {ra["aer_sampling_seconds"]:.3f}',
 '=== INSTANCE B (OFFICIAL 3x3) ===',
 f'CP-SAT exact makespan: {references["B"]["objective"]} h (OPTIMAL)',
 f'QUBO variables/interactions: {MB["variables"]}/{MB["interactions"]}',
 f'QUBO solver status: {rb["status"]}; warm-start used: {rb["warmstart_used"]}; best energy: {rb["best"]["energy"] if rb["best"] else "NONE"}',
 f'BQM solver seconds: {rb["wall_seconds"]:.3f}',
 '=== STAGE 2 (PROPOSED EXTENSION) ===',
 f'Downtime: milling [3,5); due dates: Z1=7, Z2=5, Z3=10; weights: 1,3,1',
 f'Objective: F = makespan + 2*weighted_tardiness; CP-SAT exact optimal F: {references["Stage2"]["objective"]}',
 f'QUBO variables/interactions: {ME["variables"]}/{ME["interactions"]}',
 f'QUBO solver status: {re["status"]}; warm-start used: {re["warmstart_used"]}',
 f'QUBO decoded result: {json.dumps(re["best"]["objective"]) if re["best"] else "NONE"}',
 f'BQM solver seconds: {re["wall_seconds"]:.3f}',
 '=== STAGE 2 ENERGY-AWARE (PROPOSED; NOT OFFICIAL) ===',
 'Synthetic power T/F/S = 8/12/6 kW; tariff 11c offpeak, 27c peak [2,6).',
 'Objective: makespan + 2*weighted_tardiness + 0.02*electricity_cost_USD',
 f'QUBO model variables/interactions: {MENERGY["variables"]}/{MENERGY["interactions"]}; added variables vs Stage2: {MENERGY["variables"]-ME["variables"]}',
 f'Exact CP-SAT global optimum F: {references["Stage2Energy"]["objective"]:.6f}; status: {references["Stage2Energy"]["status"]}',
 f'QUBO solver status: {energy_run["status"]}; warm-start used: {energy_run["warmstart_used"]}',
 f'QUBO result F: {energy_run["best"]["objective"]["F"] if energy_run["best"] else "NONE"}; global optimum matched: {energy_pass}',
 f'QUBO energy, recomputed: {energy_run["best"]["energy"] if energy_run["best"] else "NONE"}',
 f'Energy cost USD / peak energy kWh: {energy_run["best"]["objective"]["electricity_cost_usd"] if energy_run["best"] else "NONE"} / {energy_run["best"]["objective"]["peak_energy_kwh"] if energy_run["best"] else "NONE"}',
 f'QUBO solver seconds: {energy_run["wall_seconds"]:.3f}',
 f'Energy-cost comparison (baseline minus energy-aware), USD: {energy_comparison["energy_cost_delta_usd_baseline_minus_energy"]}',
 '=== ACCEPTANCE ===',
 json.dumps(qualifications,sort_keys=True),
 '=== EVIDENCE ===',
 'All execution data, circuits, actual bitstrings, schedules, model coefficients, CP-SAT certificates, figures and evidence manifest are included in ZIP.',
 'DO NOT imply physical QPU or VeloxQ jobs were performed. Do not call neal a quantum processor.',
 'Human PM still needs public GitHub repository, actual 5-minute demo video URL, submission form, and live 3-minute presentation.'
]
write_text('FINAL_RESULTS_FOR_CHATGPT.txt','\n'.join(summary))
print('\n'.join(summary))

if prior_schedule_energy and energy_best:
    fig,ax=plt.subplots(figsize=(7,4))
    costs=[prior_schedule_energy['electricity_cost_usd'],
           energy_best['objective']['electricity_cost_usd']]
    ax.bar(['Downtime/deadline QUBO','Energy-aware QUBO'],costs)
    ax.set_ylabel('Electricity cost (USD)')
    ax.set_title('Same synthetic tariff; independently decoded schedules')
    fig.tight_layout()
    fig.savefig(ROOT/'figures/Stage2_energy_cost_comparison.png',dpi=160)
    plt.close(fig)


## Cell 10 — Auto-generate required documentation, evidence hashes, final ZIP, download

This produces the public-repository supporting documents as drafts based on **actual recorded results**. The Human PM must review and publish these, record an actual demo video, submit the repository URL, and present live. A readiness failure is disclosed rather than hidden.

In [ ]:
now=datetime.now(timezone.utc).isoformat()
project_md=f'''# Project Design — ERP Quantum Production Scheduling (2026)

## Decision problem
Machine order T→F→S, known integer processing times, one operation start per job-machine, no machine overlaps, minimize makespan. Official Instances A (2×2) and B (3×3) are supplied in inputs/.

## Time-indexed QUBO
Binary x[j,m,t] selects operation start t. Fixed-horizon energy: P * sum_(j,m) (1-sum_t x[j,m,t])^2 + P * sum_(conflicting machine starts) x_a*x_b + P * sum_(invalid job-order starts) x_a*x_b. Horizons H=4 (A) and H=8 (B). Candidate time windows calculated by prefix/suffix job durations. In Stage 2, add exact outage start pruning and linear weighted tardiness on final-operation x variables. In the synthetic energy extension, per-operation electricity cost is another linear one-hot start coefficient; no additional start binary variables or quadratic interactions are needed at fixed H=9.

Penalty A=1, B=5, Stage2=100. Number of variables/unique quadratic interactions: A {MA['variables']}/{MA['interactions']}; B {MB['variables']}/{MB['interactions']}; extension {ME['variables']}/{ME['interactions']}; energy extension {MENERGY['variables']}/{MENERGY['interactions']}.

Baseline QUBOs search for feasibility at chosen horizons. A global makespan optimum is proven independently with OR-Tools CP-SAT, not by a failed or successful heuristic sample alone.

## Solvers actually used
Stage A: QAOA state-vector expected-energy optimization (Qiskit Statevector + SciPy COBYLA); final circuit **executed on Qiskit Aer** with actual measurement shots. Stage B and both extensions: neal **classical simulated annealing** on a binary quadratic model. Classical exact reference: OR-Tools CP-SAT. No physical QPU or VeloxQ executed.

## Proposed Stage 2
Outage on milling interval [3,5); deadlines Z1=7,Z2=5,Z3=10; priority weights Z1=1,Z2=3,Z3=1; F=makespan+2*weighted tardiness. This is investigator-designed, not official input. Fixed-horizon H=9 QUBO optimizes weighted tardiness while the *complete* F objective is separately optimized and certified using CP-SAT. Synthetic energy extension: power T/F/S=8/12/6kW, peak hours [2,6), off-peak/peak 11/27 cents per kWh; weighted energy coefficient 0.02 USD. Each selected start variable carries exact tariff charge as a linear coefficient, preserving 40 variables. CP-SAT independently optimizes full F including makespan, weighted tardiness and electricity cost.

## Results and limitations
A: Aer QAOA status {ra['status']}; exact reference 4h; optimal shot frequency {ra['optimal_shot_fraction']:.6f}.
B: QUBO status {rb['status']}; exact reference 8h; warm-start used={rb['warmstart_used']}.
Stage2: QUBO status {re['status']}; CP-SAT exact F=11; warm-start used={re['warmstart_used']}.
Energy Stage2: classical QUBO status {energy_run['status']}; CP-SAT exact full F={references['Stage2Energy']['objective']:.6f}; QUBO matched exact={energy_pass}; warm-start={energy_run['warmstart_used']}.
A simulator is not quantum-hardware execution, and a classical BQM optimizer is not quantum advantage. Solver evidence is in outputs, including any recorded failure.
'''
write_text('repository/PROJECT_DESIGN.md',project_md)
log=f'''# Development Log (generated evidence; Human PM should add genuine dated development decisions)

## {START_UTC} — Colab execution commenced
Automatically generated organizer-defined official 2x2/3x3 inputs and proposed Stage2 extension. Installed numerical and quantum packages; recorded versions and SHA-256 evidence.

## {now} — Executed models and validation
A: CP-SAT OPTIMAL 4h; QAOA actual Aer status={ra['status']}, shots={ra['shots']}, optimal count={ra['optimal_shots']}.
B: CP-SAT OPTIMAL 8h; classical QUBO status={rb['status']}, warmstart={rb['warmstart_used']}.
Stage2: CP-SAT OPTIMAL F=11; classical QUBO status={re['status']}, warmstart={re['warmstart_used']}.

## Engineering decisions
Fixed-horizon time-indexed QUBO; start-window pruning; penalty strengths 1/5/100 respectively; Aer for mandatory gate-model algorithm; neal for QUBO; CP-SAT for independent exact optimum. Any failed solver samples retained in JSON and not misrepresented.

## Required Human PM additions
Document actual manual design/revision decisions and problem fixes. Do not invent earlier experimental runs.
'''
write_text('repository/DEVELOPMENT_LOG.md',log)
readme='''# QPoland 2026 — ERP Scheduling, AI + Quantum

Open `ERP_Quantum_Hackathon_Option_A_Upgraded_Energy_Colab.ipynb` in Google Colab and select **Runtime → Run all** (CPU runtime). The notebook installs all dependencies, includes official problem inputs, performs the QAOA/Aer simulator run, runs binary QUBO optimization using neal, certifies exact references via OR-Tools CP-SAT, executes both a proposed downtime/deadline Stage2 and an additional synthetic time-of-use energy Stage2, and downloads `ERP_Quantum_Hackathon_Evidence.zip`.

## Required exported files
`schedules/schedule_2x2.json` and `schedules/schedule_3x3.json` exist only when actual sampler-generated schedules pass validation. Copy them to repository root as `schedule_2x2.json` and `schedule_3x3.json` for official submission. The notebook archive includes `inputs/`, `qubos/`, `quantum/`, `verification/`, `figures/`, `reports/`, and Stage2 outputs. `FINAL_RESULTS_FOR_CHATGPT.txt` contains compact numbers for writing final materials.

## Three-party co-pilot / AI usage
Human Expert/Project Manager sets research objectives, approves experiments, reviews and accepts results, controls public submissions; ChatGPT acting as Agentic AI Project Coordinator assisted with data interpretation, mathematical formulation, QUBO design, Python/Qiskit code generation, automated checks, scientific reporting, and presentation drafts; Google Colab with Qiskit Aer and classical solvers performs actual computations. The team independently examines evidence and assumes responsibility for claims. Do not claim physical QPU/VeloxQ usage unless independently executed later and supported by logs.

## Methods and limitations
A is real Qiskit Aer QAOA circuit measurement, parameter-fitted using noiseless state-vector + SciPy. B and extension use neal, a **classical** QUBO sampler, potentially with an explicitly marked CP-SAT warm-start. CP-SAT is used for exact verification, not disguised as a quantum optimizer. Stage2 outage/deadline and electricity parameters are separately labeled synthetic research extensions; there is NO implemented peak-energy hard cap, machine-flexibility model, operator pool, or sequence-dependent setup constraint in this notebook.

## Needed before final submission
(1) Add actual notebook file and verified JSON files to the public MIT-licensed repo; (2) audit all actual results; (3) upload and link recorded demonstration video; (4) submit the organizer's Google Form; (5) prepare for 3-minute live presentation and Q&A. The notebook cannot execute these Human PM actions.
'''
write_text('repository/README.md',readme)
write_text('repository/AI_USAGE.md','Human PM sets and approves objectives, executes/reviews workflows, and accepts final responsibility. ChatGPT acts as Agentic AI Project Coordinator to design mathematics, write and audit code, and produce documentation. Google Colab Qiskit Aer, neal, and OR-Tools execute the actual calculations. Actual simulator, classical solver, and hypothetical extension data are distinguished. No physical quantum hardware or VeloxQ execution claimed.')
video=f'''# Demo video guide (maximum five minutes; NOT a completed recording)
0:00–0:30: Official MetalTech jobs and makespan.
0:30–1:30: Time-indexed QUBO, penalty terms, 72 naive vs 39 reduced variables.
1:30–2:30: Show Colab actual QAOA/Aer output (status {ra['status']}, shots {ra['shots']}) and B classical QUBO output ({rb['status']}). Do not describe neal as physical quantum computing.
2:30–3:30: Synthetic downtime/deadline extension (CP-SAT F=11) and energy-aware extension (CP-SAT exact F={references['Stage2Energy']['objective']:.5f}); report actual sampler status and honest warm-start flags.
3:30–4:30: Independent schedule checker, exact references, measurement and runtime evidence, QUBO coefficients.
4:30–5:00: Claim only supported outcomes and explain limitations.
Human PM: capture actual Colab execution screen, record narration, upload unlisted video, insert URL in README.
'''
write_text('repository/DEMO_VIDEO_GUIDE.md',video)
write_text('repository/LIVE_PRESENTATION_GUIDE.md',
 '3-minute live demo (guide, NOT a completed presentation): 0:00-0:30 problem; 0:30-1:15 encoding and QAOA; 1:15-2:00 live notebook result and checks; 2:00-2:40 Stage2 energy-cost/no-added-binaries extension and classical comparison; 2:40-3:00 conclusion. Then approx. two minutes Q&A. Prepare to discuss feasibility penalties, window reduction, actual Aer vs neal, CP-SAT certificates, and any warm start.')
write_text('repository/SUBMISSION_CHECKLIST.md',
 f'''# Submission readiness — {now}
- [ ] Public GitHub MIT repository including notebook code
- [{'x' if qualifications['A_qaoa_optimal'] else ' '}] Real Qiskit Aer QAOA result for A + validated schedule_2x2.json
- [{'x' if qualifications['B_qubo_optimal'] else ' '}] Real neal QUBO result for B + validated schedule_3x3.json
- [{'x' if qualifications['Stage2_global_optimal'] else ' '}] Proposed Stage2 downtime/deadlines with independent proof
- [{'x' if qualifications['Stage2Energy_global_optimal'] else ' '}] Proposed energy-aware Stage2 matched CP-SAT full-objective certificate
- [x] Automated development log draft, Project Design draft, README, AI-use disclosure
- [ ] Human PM review and finalize documents, actual demonstration video URL (max 5 minutes)
- [ ] Submit official form before October 18, 2026, 15:00 UTC+2
- [ ] Deliver 3-minute live presentation + Q&A on October 18
''')
write_text('repository/LICENSE',
'''MIT License

Copyright (c) 2026 The ERP Quantum Hackathon project contributors

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
''')
# Hash all files except manifest to avoid self-referential hashing
manifest={}
for p in sorted(ROOT.rglob('*')):
    if p.is_file():manifest[str(p.relative_to(ROOT))]={'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size}
write_json('manifest.json',{'created_utc':now,'file_count':len(manifest),'files':manifest,'run_gates':qualifications,
  'data_integrity':'sha256 over content of each evidence file; manifest itself excluded',
  'physical_qpu_executed':False})
archive=Path('/content/ERP_Quantum_Hackathon_Evidence.zip') if Path('/content').exists() else Path.cwd()/'ERP_Quantum_Hackathon_Evidence.zip'
if archive.exists():archive.unlink()
shutil.make_archive(str(archive.with_suffix('')),'zip',root_dir=ROOT.parent,base_dir=ROOT.name)
print('Evidence bundle ready:',archive,'size bytes',archive.stat().st_size)
try:
    from google.colab import files
    files.download(str(archive))
    print('Browser evidence ZIP download requested.')
except ImportError:
    print('Not Colab: evidence ZIP retained at',archive)
except Exception as err:
    print('Browser download could not start. ZIP is saved at',archive,'error:',type(err).__name__,str(err))

write_text('repository/ENERGY_EXTENSION_METHOD.md', f"""# Synthetic time-of-use energy scheduling extension

## Real-world question
Can MetalTech shift manufacturing operations outside an expensive tariff window while respecting job due dates, machine precedence, downtime, and a short makespan? This is a **proposed Stage 2 research scenario**, not data from the official challenge.

## What distinguishes the implementation
The time-indexed QUBO at fixed H=9 uses one-hot operation-start variables. Adding electricity cost requires **zero additional binary variables** and **zero additional quadratic interactions**: each start decision has a known hour-overlap energy charge and receives an extra linear coefficient. This computationally efficient modeling tactic is not claimed to be a new theoretical invention.

## Explicit synthetic assumptions
Machine kW T/F/S=8/12/6. Peak tariff: 27 cents/kWh during hours [2,6); offpeak 11 cents/kWh. Hours are integer and durations fixed. Electricity cost is in USD and contributes 0.02 per USD to F. This example does not implement a kWh cap, operator scheduling, or machine setup times.

## Objective and solver separation
QUBO fixed H=9 minimizes `2*weighted tardiness + 0.02*electricity_cost_USD`; it does not contain an explicit Cmax term. For each decoded sample, we calculate `F=Cmax+2*weighted_tardiness+0.02*electricity_cost_USD`. Independent OR-Tools CP-SAT minimizes the **full** F on the original job/machine constraints, downtime, and tariff model. Heuristic solutions are globally certified only when they match that exact reference.

## Audited results (from actual execution)
QUBO run status: {energy_run['status']}
Warm-start used: {energy_run['warmstart_used']}
Exact CP-SAT F: {references['Stage2Energy']['objective']:.6f}
Best decoded QUBO F: {energy_best['objective']['F'] if energy_best else 'NO FEASIBLE SAMPLE'}
Global optimum match: {energy_pass}
QUBO variables: {MENERGY['variables']}; quadratic interactions: {MENERGY['interactions']}

The `.json` logs, complete coefficient list, and audited schedule are preserved elsewhere in the archive.
""")

## Human PM handoff — Only essential information to paste back

1. Locate `ERP_QUANTUM_EVIDENCE/FINAL_RESULTS_FOR_CHATGPT.txt` inside the downloaded ZIP, and paste its entire text into our chat. Do **not** paste large circuit or JSON logs unless I request them.
2. Keep the entire ZIP and the notebook for the public GitHub submission and scientific auditing. The ZIP includes draft `README.md`, `PROJECT_DESIGN.md`, `DEVELOPMENT_LOG.md`, `AI_USAGE.md`, MIT license, and presentation outlines.
3. The video URL, final public repository URL, submission form, and actual live presentation remain Human PM actions. No video or live presentation has been recorded by this notebook.
4. If a cell fails, copy its traceback and stop; do not call it a successful experiment. Share that error for a complete notebook correction.
The energy-aware model is a **synthetic optional Stage 2 experiment** and is reported independently of the mandated A/B instances. Preserve the whole evidence ZIP even if an optional comparison fails.